# 1. **ติดตั้ง Library & เตรียมชุดข้อมูล (Setup)**

In [ ]:
!pip install -qU qdrant-client langchain langchain-community langchain-huggingface llama-index llama-index-vector-stores-qdrant llama-index-embeddings-huggingface dspy-ai sentence-transformers

In [ ]:
documents = [
    "LLM (Large Language Model) คือโมเดลที่ฝึกฝนด้วยข้อความจำนวนมหาศาล มีหลักการทำงานคือการคาดเดาคำถัดไป",
    "Prompt Engineering คือศาสตร์ในการออกแบบและปรับแต่งคำสั่ง เพื่อควบคุมผลลัพธ์ของ AI ให้แม่นยำ",
    "DSPy คือเฟรมเวิร์กสำหรับเขียนโปรแกรมและปรับจูนโมเดลภาษาอัตโนมัติ โดยไม่ต้องนั่งเดา Prompt เอง",
    "Vector Database ใช้สำหรับเก็บข้อมูลในรูปแบบเวกเตอร์ เพื่อการค้นหาด้วยความหมาย (Semantic Search)",
    "การจูนโมเดล (Model Tuning) ช่วยเพิ่มความแม่นยำเฉพาะทางด้วยเทคนิคอย่าง PEFT หรือ LoRA"
]

# 3. กำหนดชื่อ Embedding Model (ตัวเบา รองรับภาษาไทย)
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("✅ ติดตั้งแพ็กเกจและเตรียมข้อมูลเสร็จสิ้น!")

## **LlamaIndex**

In [ ]:
from llama_index.core import VectorStoreIndex, Document
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import qdrant_client

print("--- 🟠 ทดสอบ Retrieval ด้วย LlamaIndex ---")

# 1. โหลด Embedding Model
li_embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

# 2. เตรียม Qdrant Client และ Vector Store
li_client = qdrant_client.QdrantClient(location=":memory:")
li_vector_store = QdrantVectorStore(client=li_client, collection_name="llamaindex_demo")

# 3. แปลง Text เป็น Document และสร้าง Index
li_docs = [Document(text=doc) for doc in documents]
li_index = VectorStoreIndex.from_documents(
    li_docs, 
    vector_store=li_vector_store, 
    embed_model=li_embed_model
)

# 4. ทดสอบดึงข้อมูลผ่าน Retriever
li_retriever = li_index.as_retriever(similarity_top_k=1)
query_2 = "การหาข้อมูลด้วยความหมายและตัวเลข"
print(f"คำค้นหา: '{query_2}'")

li_results = li_retriever.retrieve(query_2)
print(f"ผลลัพธ์ : {li_results[0].node.text}\n")

## **DSPY**

In [ ]:
import dspy
from dspy.retrieve.qdrant_rm import QdrantRM
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, Distance, VectorParams

print("--- 🟢 ทดสอบ Retrieval ด้วย DSPy ---")

# 1. โหลด Embedding Model ฝั่ง Python ธรรมดาเพื่อแปลงเวกเตอร์
model = SentenceTransformer(EMBED_MODEL_NAME)
vectors = model.encode(documents)

# 2. เตรียมฐานข้อมูล Qdrant เอง (เนื่องจาก DSPy หน้าที่หลักคือ "ดึง" ไม่ใช่ "ยัด" ข้อมูล)
dspy_client = QdrantClient(location=":memory:")
dspy_client.recreate_collection(
    collection_name="dspy_demo",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# ใส่ข้อมูลเข้าไปพร้อม Vector
points = [
    PointStruct(id=i, vector=vectors[i].tolist(), payload={"document": documents[i]})
    for i in range(len(documents))
]
dspy_client.upsert(collection_name="dspy_demo", points=points)

# 3. ตั้งค่าให้ DSPy รู้จัก Qdrant (เราเขียนฟังก์ชัน Custom Vectorizer เพื่อแปลงคำค้นหาเป็นเวกเตอร์)
def custom_vectorizer(texts):
    if isinstance(texts, str): texts = [texts]
    return model.encode(texts).tolist()

qdrant_retriever = QdrantRM(
    qdrant_client=dspy_client, 
    qdrant_collection_name="dspy_demo", 
    document_field="document", # ฟิลด์ที่เก็บ Text จริงๆ
    vectorizer=custom_vectorizer
)

# ประกาศให้ระบบ DSPy ใช้ Retriever ตัวนี้เป็นค่า Default
dspy.settings.configure(rm=qdrant_retriever)

# 4. ทดสอบใช้งาน (สามารถเอา dspy.Retrieve ไปประกอบใน Module ได้เลย)
query_3 = "อยากทำโมเดลเฉพาะทางแบบประหยัดแรง"
print(f"คำค้นหา: '{query_3}'")

dspy_results = dspy.Retrieve(k=1)(query_3)
print(f"ผลลัพธ์ : {dspy_results.passages[0]}\n")

## **LangChain**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant

print("--- 🔵 ทดสอบ Retrieval ด้วย LangChain ---")

# 1. โหลด Embedding Model
lc_embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL_NAME)

# 2. สร้าง Qdrant Vector Store และยัดข้อมูลลงไป (แบบ In-Memory)
lc_qdrant = Qdrant.from_texts(
    documents,
    lc_embeddings,
    location=":memory:", 
    collection_name="langchain_demo"
)

# 3. ทดสอบดึงข้อมูล (Retrieve)
query_1 = "เครื่องมือที่ช่วยจัดการคำสั่ง AI ให้เป็นระบบ"
print(f"คำค้นหา: '{query_1}'")

lc_results = lc_qdrant.similarity_search(query_1, k=1)
print(f"ผลลัพธ์ : {lc_results[0].page_content}\n")